<a href="https://colab.research.google.com/github/ISPC-PROYECTOS/CienciaDatosABP/blob/main/Data_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importamos el dataset de Kaggle, buscamos el archivo CSV

In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("lucalullo/global-ghg-gas-emissions-by-country-1950-2024")
print("Ruta del dataset:", path)

archivos = os.listdir(path)
print("Archivos descargados:", archivos)

100%|██████████| 299k/299k [00:00<00:00, 47.4MB/s]

Extracting files...
Ruta del dataset: /root/.cache/kagglehub/datasets/lucalullo/global-ghg-gas-emissions-by-country-1950-2024/versions/1
Archivos descargados: ['emissioni-gas-serra-globali-per-paese-1950-2024.csv']


Buscamos el primer archivo que termine en .csv y unimos la ruta de la carpeta con el nombre del archivo. Luego lo cargamos en pandas y verificamos que se haya cargado.

In [2]:
import pandas as pd

# Buscamos el primer archivo que termine en '.csv'
nombre_csv = [archivo for archivo in archivos if archivo.endswith('.csv')][0]

# Unimos la ruta de la carpeta con el nombre del archivo
ruta_completa = os.path.join(path, nombre_csv)

# Lo cargamos en pandas
df = pd.read_csv(ruta_completa)

# Vemos las primeras 5 filas para confirmar que todo salió bien
df.head()

,country,year,iso_code,total_ghg,ghg_per_capita
0,Afghanistan,1950,AFG,19.868742,2.555078
1,Afghanistan,1951,AFG,21.069101,2.673967
2,Afghanistan,1952,AFG,22.094320,2.766014
3,Afghanistan,1953,AFG,23.255630,2.872235
4,Afghanistan,1954,AFG,24.250988,2.954572


# **Primer análisis del dataset**
Este conjunto de datos contiene las emisiones de gases de efecto invernadero a nivel nacional desde 1950 hasta 2024.

El conjunto de datos se deriva del conjunto de datos de emisiones de CO₂ y gases de efecto invernadero publicado por Our World in Data, que integra varias fuentes científicas ampliamente utilizadas en la investigación climática.

Cada observación representa un registro de un país y un año, y abarca 199 países de todo el mundo.

Incluye información sobre:
- Emisiones totales de gases de efecto invernadero
- Emisiones de gases de efecto invernadero per cápita

**Variables**

- Country → nombre del país
- Year → año de observación
- Iso_code → Código de país ISO
- Total_ghg → emisiones totales de gases de efecto invernadero expresadas en millones de toneladas de CO₂ equivalente
- Ghg_per_capita → emisiones de gases de efecto invernadero por persona, expresadas en toneladas de CO₂ equivalente per cápita.

El dataset posee un total de 14.925 registros

**Fuentes**

- Dataset “Global GHG gas emissions by country (1950–2024)” publicado por Luca Lullo en Kaggle.
- Datos originales derivados de Our World in Data, que integran:

   - Global Carbon Project - Global Carbon Budget
   - PRIMAP-hist greenhouse gas emissions database
   - United Nations World Population Prospects
   - Gapminder population series


# **Análisis de datos inicial**
Vamos a explorar el df para entender su estructura, identificar valores nulos, duplicados y obtener estadísticas descriptivas.

In [3]:
# 1. Información general del DataFrame y tipos de datos
print('Información general del DataFrame:')
df.info()

# 2. Conteo de valores nulos por columna
print('\nConteo de valores nulos por columna:')
display(df.isnull().sum())

Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14925 entries, 0 to 14924
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         14925 non-null  object 
 1   year            14925 non-null  int64  
 2   iso_code        14925 non-null  object 
 3   total_ghg       14475 non-null  float64
 4   ghg_per_capita  14475 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 583.1+ KB

Conteo de valores nulos por columna:


,0
country,0
year,0
iso_code,0
total_ghg,450
ghg_per_capita,450


El dataset muestra un total de 14.925 registros distribuidos en cinco columnas. Las variables categóricas y la temporal están completas, pero las métricas de emisiones (total_ghg y ghg_per_capita) presentan 450 valores faltantes cada una (un 3 % del total). Esto indica que, aunque la mayoría de este dataset es consistente, será necesario aplicar técnicas de tratamiento de datos faltantes para garantizar la validez de futuros análisis.

In [4]:
# 3. Conteo de filas duplicadas
num_duplicados = df.duplicated().sum()
print(f'Número de filas duplicadas: {num_duplicados}')

if num_duplicados > 0:
    print('\nEjemplo de filas duplicadas:')
    display(df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()).head()) # Muestra algunas filas duplicadas

Número de filas duplicadas: 0


El análisis de duplicados muestra que el dataset no tiene filas repetidas, confirmando que son registros únicos y no será necesario aplicar técnicas de limpieza relacionadas con duplicados.

In [5]:
# 4. Parámetros estadísticos descriptivos para columnas numéricas
print('\nParámetros estadísticos descriptivos:')
display(df.describe())


Parámetros estadísticos descriptivos:


,year,total_ghg,ghg_per_capita
count,14925.000000,14475.000000,14475.000000
mean,1987.000000,188.028820,9.259861
std,21.649436,706.305423,12.160517
min,1950.000000,-9.218572,-5.197811
25%,1968.000000,7.289340,2.858160
50%,1987.000000,34.407463,5.909035
75%,2006.000000,99.218967,11.049553
max,2024.000000,14107.006836,369.705292


Las columnas numéricas muestran que la variable "year" abarca desde 1950 hasta 2024. Las variables de emisiones (total_ghg y ghg_per_capita) muestran una gran variabilidad entre países y años; estos parámetros permiten identificar tendencias centrales y la dispersión de los datos.

In [6]:
# 5. Parámetros estadísticos descriptivos para columnas categóricas (objetos)
print('\nParámetros estadísticos descriptivos para columnas categóricas:')
display(df.describe(include='object'))


Parámetros estadísticos descriptivos para columnas categóricas:


,country,iso_code
count,14925,14925
unique,199,199
top,Afghanistan,AFG
freq,75,75


El dataset muestra 199 países distintos. Cada país cuenta con aproximadamente 75 registros, lo que indica la presencia de múltiples observaciones por país a lo largo del rango temporal.

# Limpieza y estandarización del dataset
Verificamos si es necesario aplicar algun método de limpieza o estandarización para poder trabajar con los datos.

In [12]:
#Quitamos los espacios iniciales y finales, y cambiamos los objetos a minúsculas
df['country'] = df['country'].str.strip().str.lower()
df['iso_code'] = df['iso_code'].str.strip().str.lower()

#Listamos los países para verificar si alguno se encuentra escrito de diferente manera
columnas_anormalizar = ["country", "iso_code"]
for columna in columnas_anormalizar:
    print("------")
    print(df[columna].value_counts(dropna=False).to_string())


------
country
afghanistan                         75
albania                             75
algeria                             75
andorra                             75
angola                              75
antigua and barbuda                 75
argentina                           75
armenia                             75
australia                           75
austria                             75
azerbaijan                          75
bahamas                             75
bahrain                             75
bangladesh                          75
barbados                            75
belarus                             75
belgium                             75
belize                              75
benin                               75
bhutan                              75
bolivia                             75
bosnia and herzegovina              75
botswana                            75
brazil                              75
brunei                              75
bulgaria  

El dataset no posee ni nombre del país ni código ISO que se encuentre escrito de una manera diferente.

En las columnas numéricas, el año de observación es de tipo int64 lo cual es correcto ya que todos son valores enteros. Las columnas de emisión total y de emisión per capita son valores float, lo cual tambien corresponde a los valores contenidos. No es necesaria transformación de datos.

Trabajamos los valores nulos de emisión total y de emisión per capita. Para esto procederemos primero a identificar los países que no poseen datos registrados y los eliminamos.
Los motivos para ésto son:


1.   Cero información: Al tener un 100% de nulos, no aportan nada a promedios, tendencias o sumatorias. Son "filas vacías" para el análisis de emisiones.
2.   Imputación imposible: No podemos "rellenar" los datos (imputación) basándonos en años anteriores o posteriores porque no existe ningún punto de referencia en todo el histórico de esos países.
3.   Limpieza estadística: Si calculamos, por ejemplo, el "Promedio de emisiones por país", tener estos países en la lista nos obligaría a estar filtrando nulos constantemente para evitar errores o resultados NaN.
4.   Impacto mínimo: Eliminar estos 6 países solo reduce el dataset en 450 filas de un total de casi 15,000 (Aproximadamente 3%), pero mejora significativamente la "salud" de las estadísticas.



In [14]:
# Identificamos países que tienen TODOS sus valores de total_ghg como nulos
paises_sin_datos = df.groupby('country')['total_ghg'].apply(lambda x: x.isnull().all())
nombres_paises_vacios = paises_sin_datos[paises_sin_datos].index.tolist()

print(f"Países eliminados por falta total de datos: {nombres_paises_vacios}")

# Filtramos el DataFrame original para conservar solo los países que SÍ tienen datos
df_limpio = df[~df['country'].isin(nombres_paises_vacios)].copy()

# Verificamos resultados
print(f"Registros originales: {len(df)}")
print(f"Registros tras la limpieza: {len(df_limpio)}")

Países eliminados por falta total de datos: ['kosovo', 'marshall islands', 'monaco', 'palestine', 'san marino', 'vatican']
Registros originales: 14925
Registros tras la limpieza: 14475


Vamos a agregar una columna de cantidad estimada de población, basándonos en los datos de emisión total y emisión per capita.
En este cálculo, como las cantidades en emisión total estan expresadas en millones de toneladas de CO₂ equivalente, el resultado será a su vez en millones de habitantes.

In [16]:
#Creamos la nueva columna "poblacion_en_millones" y hacemos el cálculo
df_limpio['poblacion_en_millones'] = df_limpio['total_ghg'] / df_limpio['ghg_per_capita']

# Verificamos los resultados
print(df_limpio[['country', 'year', 'poblacion_en_millones']].head())

       country  year  poblacion_en_millones
0  afghanistan  1950               7.776180
1  afghanistan  1951               7.879343
2  afghanistan  1952               7.987784
3  afghanistan  1953               8.096703
4  afghanistan  1954               8.207954


# Dataset final

In [17]:
#Exportamos el dataset resultante luego de la limpieza y estandarización
df_limpio.to_csv('dataset_final.csv', index=False)
